In [3]:
from importlib import import_module
from time import process_time
from typing import Any

import torch
from sympy import sequence
from torch._dynamo.variables import nn_module
from torch.fx.experimental.migrate_gradual_types.constraint_transformation import register_transformation_rule

x = torch.randn(4, 3)
weight = torch.randn(2, 3, requires_grad=True)
bias = torch.randn(2, requires_grad=True)

y = x @ weight.T + bias

"""
    forward(): Module负责组织计算
    nn.Module 与 nn.functional 的关系
    Parameter: 需要被优化器更新的张量
    buffer: 属于模型状态,但不可以用于可学习参数
    子模块: Module 可以嵌套Module
    state_dict: 模型状态的字典
    train() 和eval()  切换模块的行为
    Lazy  Module :推迟确定的输入模块
"""



'\n    forward(): Module负责组织计算\n    nn.Module 与 nn.functional 的关系\n    Parameter: 需要被优化器更新的张量\n    buffer: 属于模型状态,但不可以用于可学习参数\n    子模块: Module 可以嵌套Module\n    state_dict: 模型状态的字典\n    train() 和eval()  切换模块的行为\n    Lazy  Module :推迟确定的输入模块\n'

In [4]:
from pprint import pprint
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor

print("PyTorch version: ", torch.__version__)

in_features = 3
out_features = 2

weight = torch.randn(out_features, in_features, requires_grad=True)
bias = torch.randn(out_features, requires_grad=True)

x = torch.randn(4, in_features)
y = x @ weight.T + bias

print('y.shape: ', y.shape)

PyTorch version:  2.13.0+cpu
y.shape:  torch.Size([4, 2])


In [4]:
class SimpleLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))

    def forward(self, x):
        return x @ self.weight.T + self.bias


linear = SimpleLinear(3, 2)

for name, param in linear.parameters():
    print(f"{name},{param.size()}")


tensor([-0.9160, -0.4536,  0.7759], grad_fn=<UnbindBackward0>),torch.Size([3])
-0.1901848465204239,torch.Size([])


In [5]:
# forward(): Module负责组织计算
class SimpleLinear(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return x @ self.weight.T + self.bias


x = torch.randn(4, 3)
y = linear(x)
print('y.shape: ', y.shape)

y.shape:  torch.Size([4, 2])


In [9]:
# Parameter: 需要被优化器更新的张量

class BadLiner(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.weight = torch.randn(2, 3, requires_grad=True)


bad = BadLiner()
for name, param in bad.named_parameters():
    print(f"{name},{param}")


class GoodLiner(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(2, 3))


good = GoodLiner()
for name, param in good.named_parameters():
    print(f"{name},{param}")


# 可显式注册参数
class ExplicitLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        weight = nn.Parameter(torch.randn(out_features, in_features))
        bias = nn.Parameter(torch.randn(out_features))
        self.register_parameter('weight', weight)
        self.register_parameter('bias', bias)

    def forward(self, x: Tensor) -> Tensor:
        return F.linear(x, self.weight, self.bias)


class OptionalBiasLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.randn(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, x: Tensor) -> Tensor:
        return F.linear(x, self.weight, self.bias)





weight,Parameter containing:
tensor([[-0.6673,  0.8172, -0.3763],
        [ 1.4386, -1.0549, -0.8650]], requires_grad=True)


In [13]:
# buffer: 属于模型状态,但不可以用于可学习参数

class Normalize(nn.Module):
    def __init__(self, mean: Tensor, std: Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, x: Tensor) -> Tensor:
        return (x - self.mean) / self.std


mean = torch.tensor([0.5, 0.5, 0.5])
std = torch.tensor([0.2, 0.2, 0.2])
normalize = Normalize(mean, std)

print('*' * 20 + 'Parameters' + '*' * 20)
for name, param in normalize.named_parameters():
    print(f'{name},{param}')

print('*' * 20 + 'Buffers' + '*' * 20)
for name, buffer in normalize.named_buffers():
    print(f'{name},{buffer}')


********************Parameters********************
********************Buffers********************
mean,tensor([0.5000, 0.5000, 0.5000])
std,tensor([0.2000, 0.2000, 0.2000])


In [7]:
# 子模块: Module 可以嵌套Module
class SimpleMLP(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.fc1 = nn.Linear(3, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 4)

    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


model = SimpleMLP()
print(model)

for name, param in model.named_parameters():
    print(f'{name}:{param.size()}')

for name, child in model.named_children():
    print(f"{name}:{child}")


# 模型放在一组list中

class BadStack(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.layer = [nn.Linear(3, 3), nn.Linear(3, 3)]


bad_stack = BadStack()
for name, param in bad_stack.named_parameters():
    print(f'{name},{param}')

"""
要保存一组子模块,要使用nn.ModuleList
"""


class GoodStack(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(3, 3), nn.Linear(3, 3)])

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


good_stack = GoodStack()
for name, param in good_stack.named_parameters():
    print(f'{name},{param}')

"""
对于字典模块
"""


class BadDict(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.layers = {'layer1': nn.Linear(3, 3), 'layer2': nn.Linear(3, 3)}

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers.keys():
            x = self.layers[layer](x)
        return x


bad_dict = BadDict()
for name, param in bad_dict.named_parameters():
    print(f'{name},{param}')


class GoodDict(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.layers = nn.ModuleDict({'layer1': nn.Linear(3, 3), 'layer2': nn.Linear(3, 3)})

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers.keys():
            x = self.layers[layer](x)
        return x


good_dict = GoodDict()
for name, param in good_dict.named_parameters():
    print(f'{name},{param}')

"""
模块间的执行顺序
"""

sequential_model = nn.Sequential(
    nn.Linear(3, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
)

print(sequential_model)

SimpleMLP(
  (fc1): Linear(in_features=3, out_features=8, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=8, out_features=4, bias=True)
)
fc1.weight:torch.Size([8, 3])
fc1.bias:torch.Size([8])
fc2.weight:torch.Size([4, 8])
fc2.bias:torch.Size([4])
fc1:Linear(in_features=3, out_features=8, bias=True)
relu:ReLU()
fc2:Linear(in_features=8, out_features=4, bias=True)
layers.0.weight,Parameter containing:
tensor([[ 0.5648, -0.1680,  0.0798],
        [ 0.4783,  0.4779, -0.4843],
        [ 0.4948, -0.4752,  0.5537]], requires_grad=True)
layers.0.bias,Parameter containing:
tensor([ 0.5678, -0.4454, -0.1831], requires_grad=True)
layers.1.weight,Parameter containing:
tensor([[-0.3402, -0.0130,  0.2957],
        [-0.0831, -0.2269, -0.3797],
        [-0.2027,  0.5026,  0.5554]], requires_grad=True)
layers.1.bias,Parameter containing:
tensor([ 0.5720, -0.4656, -0.5073], requires_grad=True)
layers.layer1.weight,Parameter containing:
tensor([[-0.0738, -0.2303, -0.1236],
        [ 0.2750, -0.5

In [24]:
# state_dict: 模型状态的字典

state = model.state_dict()

for key, value in state.items():
    print(f'{key},{value}')

"""
保存模型
"""
torch.save(model.state_dict(), 'model.pt')

"""
重新加载模型
"""
model = SimpleMLP()
state_dict = torch.load('model.pt')
flag = model.load_state_dict(state_dict)
print("Load state dict flag:", flag)




fc1.weight,tensor([[-0.4553, -0.0102,  0.2599],
        [-0.2472,  0.4804, -0.4762],
        [-0.0130, -0.4252, -0.5103],
        [-0.5427, -0.1014,  0.2313],
        [-0.3806,  0.3716, -0.3677],
        [-0.3575, -0.0686, -0.1801],
        [-0.2585,  0.0467,  0.4210],
        [-0.3821, -0.1390, -0.1850]])
fc1.bias,tensor([ 0.4835, -0.1527, -0.5219,  0.2901,  0.1015,  0.4398,  0.3119,  0.1432])
fc2.weight,tensor([[-0.3328,  0.0720, -0.2541,  0.1310, -0.0717, -0.1135, -0.1445, -0.1097],
        [-0.1825, -0.2400,  0.2489, -0.3448, -0.0385,  0.0157,  0.2779,  0.1620],
        [-0.2610, -0.0784, -0.3402, -0.2408,  0.0148, -0.1900, -0.0336,  0.0706],
        [-0.3257, -0.0783, -0.1363,  0.2407,  0.2078, -0.1491, -0.1063,  0.1380]])
fc2.bias,tensor([-0.0480, -0.1284,  0.1383,  0.2123])
Load state dict flag: <All keys matched successfully>


In [8]:
"""
    train() 和eval()  切换模块的行为
        - train() /eval()  控制模块的行为,例如Dropout 和 BatchNorm 在训练和评估的时候的不同的计算逻辑
        -  no_grad() / inference_mode()  控制Autograd 是否记录计算图,例如在评估节点我们不需要梯度
"""

dropout = nn.Dropout(p=0.5)
x = torch.ones(5)

dropout.train()
print("Train Mode", dropout(x))

dropout.eval()
print("Eval Mode", dropout(x))

model = SimpleMLP()
print('Initial training  mode :', model.training)

model.eval()
print("After  calling eval(): ", model.training)

model.train()
print("After calling train(): ", model.training)



Train Mode tensor([2., 0., 2., 2., 2.])
Eval Mode tensor([1., 1., 1., 1., 1.])
Initial training  mode : True
After  calling eval():  False
After calling train():  True


In [11]:
""""
 Lazy  Module :推迟确定的输入模块
    -  先创建模块,但是暂时不创建完整的参数,等到第一次真实的输入时,再根据输入的形状初始化参数.(初始化参数被放到了第一次forwoard)
"""


class LazyCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.LazyConv2d(8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.LazyConv2d(16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.LazyLinear(num_classes)

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = x.flatten(start_dim=1)
        x = self.classifier(x)
        return x


lazy_cnn = LazyCNN(num_classes=10)
print(lazy_cnn)

x = torch.randn(4, 1, 28, 28)
y = lazy_cnn(x)
print(lazy_cnn.classifier)



LazyCNN(
  (features): Sequential(
    (0): LazyConv2d(0, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): LazyConv2d(0, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): LazyLinear(in_features=0, out_features=10, bias=True)
)
Linear(in_features=784, out_features=10, bias=True)


In [14]:
"""
综合以上知识点
    -  两个线性层,作为可学习的子模块
    - 一个dropout() ,用来展示训练/评估行为
    - 一个输入归一化的mean和std ,作为buffer
    - 一个不可持久的缓存mask ,作为non-persistent buffer

    parameters() 找到可学习参数
    buffers() 找到非参数状态
    modules() 找到子模块结构
    state_dict() 导出可保存的模型状态
    train() / eval() 切换模型行为
"""


class DemoNet(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, output_dim: int = 10):
        super().__init__()
        self.register_buffer("mean", torch.zeros(input_dim))
        self.register_buffer("std", torch.ones(input_dim))
        self.register_buffer(
            "cache_mask",
            torch.zeros(input_dim, dtype=torch.bool),
            persistent=False
        )

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: Tensor) -> Tensor:
        x = (x - self.mean) / self.std
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


demo = DemoNet(input_dim=3, hidden_dim=8, output_dim=2)

print('*' * 20 + "Parameters" + '*' * 20)
for name, param in demo.named_parameters():
    print(f'{name},{param.size()}')

print('*' * 20 + "Buffers" + '*' * 20)
for name, buffer in demo.named_buffers():
    print(f'{name},{buffer.size()}')

print('*' * 20 + "State dict" + '*' * 20)
for key, value in demo.state_dict().items():
    print(f'{key},{value.size()}')

print('*' * 20 + "Submodules" + '*' * 20)
for name, model in demo.named_modules():
    print(f'{name},->,{model.__class__.__name__}')






********************Parameters********************
fc1.weight,torch.Size([8, 3])
fc1.bias,torch.Size([8])
fc2.weight,torch.Size([2, 8])
fc2.bias,torch.Size([2])
********************Buffers********************
mean,torch.Size([3])
std,torch.Size([3])
cache_mask,torch.Size([3])
********************State dict********************
mean,torch.Size([3])
std,torch.Size([3])
fc1.weight,torch.Size([8, 3])
fc1.bias,torch.Size([8])
fc2.weight,torch.Size([2, 8])
fc2.bias,torch.Size([2])
********************Submodules********************
,->,DemoNet
fc1,->,Linear
dropout,->,Dropout
fc2,->,Linear
